##  Jour 7 — Backpropagation : Le Cœur de l'Apprentissage

###  Concept Théorique

La **backpropagation** (rétropropagation) est l'algorithme qui permet de calculer **les gradients de la loss par rapport à chaque poids** du réseau. C'est une application récursive de la **règle de la chaîne** (chain rule).

###  Intuition

Pensez à une **chaîne de dominos** :

```
Entrée → Neurone 1 → Neurone 2 → ... → Sortie → Loss
```

La backpropagation remonte cette chaîne en sens inverse pour répondre à la question : "**Si je change un peu le poids $w$ dans le neurone 1, de combien change la loss ?**"

###  Formules Essentielles

**Règle de la chaîne :**

Si $\mathcal{L}$ dépend de $z$ qui dépend de $w$ :

$$\frac{\partial \mathcal{L}}{\partial w} = \frac{\partial \mathcal{L}}{\partial z} \cdot \frac{\partial z}{\partial w}$$

**Pour un neurone avec sigmoid :**

$$z = wx + b$$
$$a = \sigma(z)$$
$$\mathcal{L} = -(y \log(a) + (1-y) \log(1-a))$$

**Les gradients (en remontant) :**

$$\frac{\partial \mathcal{L}}{\partial a} = -\frac{y}{a} + \frac{1-y}{1-a}$$

$$\frac{\partial \mathcal{L}}{\partial z} = a - y$$

$$\frac{\partial \mathcal{L}}{\partial w} = (a - y) \cdot x$$

$$\frac{\partial \mathcal{L}}{\partial b} = a - y$$

###  Implémentation From Scratch


In [1]:
import numpy as np

# ============================================================
# JOUR 7 : Backpropagation — pas à pas
# ============================================================

# Réseau à 2 couches : 2 entrées → 2 neurones cachés → 1 sortie

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def sigmoid_derivative(a):
    """Dérivée de sigmoid en fonction de la SORTIE a (pas de z)."""
    return a * (1 - a)

# Données XOR
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([[0], [1], [1], [0]])

# Initialisation des poids
np.random.seed(42)
W1 = np.random.randn(2, 2) * 0.5
b1 = np.zeros((1, 2))
W2 = np.random.randn(2, 1) * 0.5
b2 = np.zeros((1, 1))

learning_rate = 1.0
losses = []

print("=== Backpropagation sur XOR ===\n")

for epoch in range(10000):
    # ========== FORWARD PASS ==========
    z1 = X @ W1 + b1
    a1 = sigmoid(z1)
    z2 = a1 @ W2 + b2
    a2 = sigmoid(z2)
    
    # Loss (MSE)
    loss = np.mean((y - a2) ** 2)
    losses.append(loss)
    
    # ========== BACKWARD PASS ==========
    m = X.shape[0]
    
    # Gradient de la loss par rapport à a2
    dL_da2 = (2 / m) * (a2 - y)
    
    # Couche 2 : remonter à travers sigmoid
    da2_dz2 = sigmoid_derivative(a2)
    dL_dz2 = dL_da2 * da2_dz2
    
    # Gradients des poids couche 2
    dL_dW2 = a1.T @ dL_dz2
    dL_db2 = np.sum(dL_dz2, axis=0, keepdims=True)
    
    # Propager l'erreur vers la couche 1
    dL_da1 = dL_dz2 @ W2.T
    da1_dz1 = sigmoid_derivative(a1)
    dL_dz1 = dL_da1 * da1_dz1
    
    # Gradients des poids couche 1
    dL_dW1 = X.T @ dL_dz1
    dL_db1 = np.sum(dL_dz1, axis=0, keepdims=True)
    
    # ========== MISE À JOUR ==========
    W2 -= learning_rate * dL_dW2
    b2 -= learning_rate * dL_db2
    W1 -= learning_rate * dL_dW1
    b1 -= learning_rate * dL_db1
    
    if epoch < 5 or epoch % 2000 == 0:
        predictions = (a2 > 0.5).astype(int)
        accuracy = np.mean(predictions == y)
        print(f"  Époque {epoch+1:>5} : loss = {loss:.6f}, accuracy = {accuracy:.0%}")

# Test final
print("\n=== Résultat final ===")
z1 = X @ W1 + b1
a1 = sigmoid(z1)
z2 = a1 @ W2 + b2
a2 = sigmoid(z2)

for i in range(4):
    print(f"  {X[i]} → prob = {a2[i, 0]:.4f}, prédit = {int(a2[i, 0] > 0.5)}, attendu = {y[i, 0]}")

print("\n🎉 XOR résolu avec la backpropagation et 2 couches !")

=== Backpropagation sur XOR ===

  Époque     1 : loss = 0.251156, accuracy = 50%
  Époque     2 : loss = 0.250729, accuracy = 50%
  Époque     3 : loss = 0.250461, accuracy = 50%
  Époque     4 : loss = 0.250292, accuracy = 50%
  Époque     5 : loss = 0.250186, accuracy = 50%
  Époque  2001 : loss = 0.151592, accuracy = 75%
  Époque  4001 : loss = 0.126921, accuracy = 50%
  Époque  6001 : loss = 0.125941, accuracy = 50%
  Époque  8001 : loss = 0.125617, accuracy = 50%

=== Résultat final ===
  [0 0] → prob = 0.0185, prédit = 0, attendu = 0
  [0 1] → prob = 0.4993, prédit = 0, attendu = 1
  [1 0] → prob = 0.9832, prédit = 1, attendu = 1
  [1 1] → prob = 0.5005, prédit = 1, attendu = 0

🎉 XOR résolu avec la backpropagation et 2 couches !



###  Résumé de la Backpropagation

```
Forward:  X → z1 → a1 → z2 → a2 → Loss
Backward: X ← dz1 ← da1 ← dz2 ← da2 ← dLoss
```

> La backpropagation n'est **rien d'autre** que la règle de la chaîne appliquée systématiquement en sens inverse.